In [1]:
import pandas as pd
import plotly.express as px

telemetry = pd.read_parquet("../data/processed/telemetry.parquet")
telemetry = telemetry.sort_values(["machineID", "datetime"]).reset_index(drop=True)

# 24-hour rolling averages per machine
for col in ["vibration", "temperature", "voltage", "pressure", "rotation"]:
    telemetry[col + "_24h"] = telemetry.groupby("machineID")[col].transform(
        lambda s: s.rolling(24, min_periods=1).mean())

print(telemetry.shape)
print(telemetry.columns.tolist())

(876100, 17)
['datetime', 'machineID', 'voltage', 'rotation', 'pressure', 'vibration', 'hours_this_year', 'model', 'age', 'operating_hours', 'temperature', 'fail_within_24h', 'vibration_24h', 'temperature_24h', 'voltage_24h', 'pressure_24h', 'rotation_24h']


In [2]:
features = ["vibration_24h", "temperature_24h", "voltage_24h", "pressure_24h",
            "rotation_24h", "age", "operating_hours"]

cutoff = pd.Timestamp("2015-09-01")
train = telemetry[telemetry["datetime"] < cutoff]
test  = telemetry[telemetry["datetime"] >= cutoff]

print("Training rows:", len(train), "| failure rows:", int(train["fail_within_24h"].sum()))
print("Testing rows: ", len(test),  "| failure rows:", int(test["fail_within_24h"].sum()))

Training rows: 582600 | failure rows: 11794
Testing rows:  293500 | failure rows: 5407


In [3]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=50,
                               class_weight="balanced", n_jobs=-1, random_state=42)
model.fit(train[features], train["fail_within_24h"])
print("Model trained")

Model trained


In [4]:
from sklearn.metrics import classification_report, confusion_matrix

pred = model.predict(test[features])
print(classification_report(test["fail_within_24h"], pred,
                            target_names=["Normal", "Failure within 24h"]))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(test["fail_within_24h"], pred))

                    precision    recall  f1-score   support

            Normal       1.00      0.99      0.99    288093
Failure within 24h       0.59      0.99      0.74      5407

          accuracy                           0.99    293500
         macro avg       0.79      0.99      0.87    293500
      weighted avg       0.99      0.99      0.99    293500

Confusion matrix (rows = actual, columns = predicted):
[[284338   3755]
 [    31   5376]]


### Model results

A Random Forest was trained on data before 1 Sept 2015 and tested on data after it, using 24-hour average sensor readings, machine age and operating hours. Failures are rare (about 2% of readings), so class weighting was used and accuracy was ignored as a metric.

**Finding:** On unseen data the model caught 5,376 of 5,407 failure-labelled hours (recall 0.99) and missed 31. Precision is 0.59, with 3,755 false alarms, so about 4 in 10 warnings are unnecessary. In a factory, a missed failure costs far more than an extra inspection, so high recall is the right priority.

**Caveat:** The temperature column was AI-generated to rise before failures, so it makes failures easier to detect than they would be with real data. These results show that the pipeline works, not that this performance would carry over to real machines.

In [5]:
feat_real = [f for f in features if "temperature" not in f]
m2 = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=50,
                            class_weight="balanced", n_jobs=-1, random_state=42)
m2.fit(train[feat_real], train["fail_within_24h"])
print(classification_report(test["fail_within_24h"], m2.predict(test[feat_real]),
                            target_names=["Normal", "Failure within 24h"]))

                    precision    recall  f1-score   support

            Normal       1.00      0.92      0.96    288093
Failure within 24h       0.19      0.95      0.31      5407

          accuracy                           0.92    293500
         macro avg       0.59      0.94      0.64    293500
      weighted avg       0.98      0.92      0.95    293500



In [6]:
imp = pd.DataFrame({"feature": features, "importance": model.feature_importances_})
imp = imp.sort_values("importance", ascending=False)
print(imp.round(3))
px.bar(imp, x="importance", y="feature", orientation="h",
       title="Which inputs matter most for predicting failure").show()

           feature  importance
1  temperature_24h       0.471
4     rotation_24h       0.293
2      voltage_24h       0.090
0    vibration_24h       0.080
3     pressure_24h       0.051
6  operating_hours       0.012
5              age       0.003


In [7]:
test = test.copy()
test["risk"] = model.predict_proba(test[features])[:, 1]

latest = test.groupby("machineID").tail(24)
risk = latest.groupby("machineID")["risk"].mean().reset_index()
risk = risk.merge(telemetry[["machineID", "model", "age"]].drop_duplicates(), on="machineID")

risk.to_parquet("../data/processed/risk.parquet")
imp.to_parquet("../data/processed/feature_importance.parquet")

print(risk.sort_values("risk", ascending=False).head(10).round(3))

    machineID   risk   model  age
14         15  0.709  model3   14
63         64  0.523  model3   20
94         95  0.444  model2   18
89         90  0.288  model2    2
43         44  0.152  model4    7
22         23  0.109  model1   17
49         50  0.105  model4    4
62         63  0.078  model4   14
70         71  0.078  model2   18
57         58  0.060  model1    5
